# E-Perf-1: Throughput Comparison (WAFER vs Native vs eKuiper)

Pipeline A at 1000 msg/s steady. 30 runs per system.
Compares sustainable throughput across three systems on macOS shakedown.

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find the shakedown result directory
result_dirs = sorted(glob.glob('../../results/e-perf-1/shakedown-macos-*'))
if not result_dirs:
    raise FileNotFoundError('No E-Perf-1 shakedown results found')
RESULT_DIR = result_dirs[-1]  # most recent
print(f'Using: {RESULT_DIR}')

In [ ]:
def load_system_data(system):
    """Load subscriber-metadata.json from all runs for a system."""
    runs = sorted(glob.glob(f'{RESULT_DIR}/{system}/run-*'))
    records = []
    for run_dir in runs:
        meta_path = os.path.join(run_dir, 'subscriber-metadata.json')
        if not os.path.exists(meta_path):
            continue
        with open(meta_path) as f:
            d = json.load(f)
        thr_path = os.path.join(run_dir, 'throughput.csv')
        thr = 0
        if os.path.exists(thr_path):
            df = pd.read_csv(thr_path)
            if 'throughput_msg_s' in df.columns and len(df) > 0:
                thr = df['throughput_msg_s'].iloc[0]
        records.append({
            'system': system,
            'total_recorded': d.get('total_recorded', 0),
            'p50_ns': d.get('latency_p50_ns', 0),
            'p95_ns': d.get('latency_p95_ns', 0),
            'p99_ns': d.get('latency_p99_ns', 0),
            'throughput_msg_s': thr,
            'parse_errors': d.get('parse_errors', 0),
            'gaps': d.get('sequence', {}).get('total_gaps', 0),
        })
    return pd.DataFrame(records)

wafer_df = load_system_data('wafer')
native_df = load_system_data('native')
ekuiper_df = load_system_data('ekuiper')
all_df = pd.concat([wafer_df, native_df, ekuiper_df], ignore_index=True)
print(f'Loaded: wafer={len(wafer_df)}, native={len(native_df)}, ekuiper={len(ekuiper_df)} runs')

In [ ]:
# Summary statistics per system
summary = all_df.groupby('system').agg({
    'throughput_msg_s': ['median', 'mean', 'std'],
    'p50_ns': ['median'],
    'p99_ns': ['median'],
    'total_recorded': ['mean'],
    'gaps': ['sum'],
}).round(1)
print('\n=== Throughput Summary (msg/s) ===')
print(summary)

# Compute ratios
wafer_thr = wafer_df['throughput_msg_s'].median()
native_thr = native_df['throughput_msg_s'].median()
ekuiper_thr = ekuiper_df['throughput_msg_s'].median()
print(f'\n=== Throughput Ratios ===')
print(f'WAFER/native: {wafer_thr/native_thr:.3f}')
print(f'WAFER/eKuiper: {wafer_thr/ekuiper_thr:.3f}')

In [ ]:
# Box plot of throughput per system
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

systems = ['wafer', 'native', 'ekuiper']
colors = ['#2196F3', '#4CAF50', '#FF9800']

data_thr = [all_df[all_df.system==s]['throughput_msg_s'].values for s in systems]
bp1 = ax1.boxplot(data_thr, tick_labels=['WAFER', 'Native', 'eKuiper'], patch_artist=True)
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Throughput (msg/s)')
ax1.set_title('E-Perf-1: Throughput Comparison')
ax1.axhline(y=1000, color='gray', linestyle='--', alpha=0.5, label='Source rate')
ax1.legend()

# Box plot of p50 latency
data_lat = [all_df[all_df.system==s]['p50_ns'].values / 1e6 for s in systems]
bp2 = ax2.boxplot(data_lat, tick_labels=['WAFER', 'Native', 'eKuiper'], patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('p50 Latency (ms)')
ax2.set_title('E-Perf-1: p50 Latency Comparison')

plt.tight_layout()
plt.savefig('../../results/e-perf-1/throughput-comparison.png', dpi=150, bbox_inches='tight')
plt.show()